# Single-experiment hysteresis curve

Select one result file below and run all cells.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt

SINGLE_GRAIN_DIR = Path.cwd()
if not (SINGLE_GRAIN_DIR / "utils").is_dir():
    SINGLE_GRAIN_DIR = Path("python/experiments/single_grain").resolve()
sys.path.insert(0, str(SINGLE_GRAIN_DIR))

from utils.metrics import MU0
from utils.plotting import save_figure
from utils.results import load_result

In [ ]:
# Change this path to plot a different experiment.
RESULT_FILE = SINGLE_GRAIN_DIR / (
    "res_fe16n2_Ms_2.4_A0_7e-12_K0_1e6/shape_cube/"
    "batch_20260617_074548_1_659460/results/"
    "single_grain_size10nm_n10_A_FS1.0e-01.npz"
)

EXPORT = True
EXPORT_DIR = SINGLE_GRAIN_DIR / "figures" / "plot_single_hysteresis"

if not RESULT_FILE.is_file():
    raise FileNotFoundError(f"Result file not found: {RESULT_FILE}")

result = load_result(RESULT_FILE)
result.path

In [ ]:
material = result.preset or result.material
dh_label = (
    f"{result.adaptive_dh_min_t:g} T"
    if result.adaptive_dh_min_t == result.adaptive_dh_min_t
    else "fixed step"
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    result.H_T,
    MU0 * result.Mz_A_per_m,
    color="black",
    linestyle="-",
    marker="*",
    linewidth=1.2,
    markersize=6,
)

if result.metrics.coercivity_status == "ok":
    coercivity_t = result.metrics.Hc_T
    ax.scatter(
        coercivity_t,
        0,
        color="red",
        marker="*",
        s=220,
        zorder=5,
    )
    ax.annotate(
        rf"$\mu_0 H_c = {coercivity_t:.3f}\ \mathrm{{T}}$",
        xy=(coercivity_t, 0),
        xytext=(14, 18),
        textcoords="offset points",
        color="red",
        fontsize=16,
        fontweight="bold",
    )
ax.axhline(0, color="0.65", linewidth=0.7)
ax.axvline(0, color="0.65", linewidth=0.7)
ax.set(
    xlabel=r"$\mu_0 H$ [T]",
    ylabel=r"$\mu_0 M_z$ [T]",
    title=f"{material} | {result.size_nm:g} nm | n={result.n} | dH={dh_label}",
)
ax.xaxis.label.set_size(20)
ax.yaxis.label.set_size(20)
ax.title.set_size(22)
ax.tick_params(axis="both", labelsize=16)
ax.grid(True, linestyle=":", alpha=0.5)
fig.tight_layout()

if EXPORT:
    figure_path = save_figure(
        fig,
        EXPORT_DIR / f"hysteresis_{result.path.stem}.png",
    )
    print(f"Saved figure to {figure_path.resolve()}")
plt.show()